In [ ]:

from jsonschema import validate
from jsonschema.exceptions import ValidationError
import json

from src.pipeline.ml.context_extractor.utils.helpers.seed_utils import enforce_reproducibility
from src.pipeline.ml.context_extractor.utils.data.data_preprocessor import prepare_data
from src.pipeline.ml.context_extractor.utils.training_pipeline_utils import train_model

import logging

logger = logging.getLogger(__name__)


activity_recognition_config_path = "config/machine-activity-recognition/machine-activity-recognition-config.json"
activity_recognition_config_schema_path = "config/machine-activity-recognition/machine-activity-recognition-config-schema.json"
        
context_extraction_config_path = "config/context-extraction/context-extraction-config.json"
context_extraction_config_schema_path = "config/context-extraction/context-extraction-config-schema.json"

spring_back_config_path = "config/springback-prediction/springback-prediction-config.json"

def get_schema_description(schema_part, path):
    """Traverse schema to find 'description' for a given path."""
    for key in path:
        if 'properties' in schema_part and key in schema_part['properties']:
            schema_part = schema_part['properties'][key]
        else:
            return None
    return schema_part.get('description')

with open(context_extraction_config_path, "r") as f:
    config = json.load(f)
with open(context_extraction_config_schema_path, "r") as f:
    schema = json.load(f)

try:
    validate(instance=config, schema=schema)
    logging.info("Configuration json is valid!")
except ValidationError as e:
    path_list = list(e.path)
    description = get_schema_description(schema, path_list)
    
    if description:
        message = f"Validation failed at '{' -> '.join(str(p) for p in path_list)}': {description}"
    else:
        message = f"Validation failed at '{' -> '.join(str(p) for p in path_list)}': {e.message}"
    
    logging.error(message)


input_path_param = config.get("inputPathParams")
preprocessing_param = config.get("preprocessingParams")
training_params = config.get("trainingParams")
occlusion_params = config.get("occlusionParams")
preprocessing_info = config.get("preprocessingParams")
seed = config.get("generalSetting").get("seed", 42)

process_part = input_path_param.get("process_part")


# ============================================================
# Seeding for reproducibility
# ============================================================
enforce_reproducibility(seed=seed)

# ============================================================
# Read and preprocess data
# ============================================================

(
X_train, Y_train, X_test, Y_test, 
springbacks_train, springbacks_test, 
experiment_configurations_train, experiment_configurations_test, 
sensor_names, target_feature_names, annot_timesteps, 
mandrel_extraction_annot_timesteps) = prepare_data(
input_path_param=input_path_param,
preprocessing_param=preprocessing_param,
)

train_model(
X_train=X_train, 
Y_train=Y_train, 
X_test=X_test, 
Y_test=Y_test,
springbacks_train=springbacks_train,
springbacks_test=springbacks_test,
experiment_configurations_train=experiment_configurations_train,
experiment_configurations_test=experiment_configurations_test, 
params=training_params,
occlusion_params=occlusion_params,
sensor_names=sensor_names,
target_feature_names=target_feature_names,
process_part=process_part,
preprocessing_info=preprocessing_info,
annot_timesteps=annot_timesteps,
mandrel_extraction_annot_timesteps=mandrel_extraction_annot_timesteps)